In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

try:
    df = pd.read_csv('teleCust.csv')
    print("Successfully loaded 'teleCust.csv'")
except FileNotFoundError:
    print("Warning: 'teleCust.csv' not found. Creating a sample dataset.")
    data = {
        'region': np.random.randint(1, 4, 1000),
        'tenure': np.random.randint(1, 72, 1000),
        'age': np.random.randint(18, 80, 1000),
        'marital': np.random.randint(0, 2, 1000),
        'address': np.random.randint(0, 50, 1000),
        'income': np.random.normal(100, 50, 1000).astype(int),
        'ed': np.random.randint(1, 5, 1000),
        'employ': np.random.randint(0, 40, 1000),
        'retire': np.random.randint(0, 2, 1000),
        'gender': np.random.randint(0, 2, 1000),
        'reside': np.random.randint(1, 6, 1000),
        'category': np.random.randint(1, 5, 1000)
    }
    df = pd.DataFrame(data)
    df.loc[df['income'] < 10, 'income'] = 10

print("\nDataset head:")
print(df.head())

if 'category' not in df.columns:
    print("\nError: Target column 'category' not found. Aborting.")
else:
    X = df.drop(['category', 'custid'], axis=1, errors='ignore')
    y = df['category']

    non_numeric_cols = X.select_dtypes(exclude=np.number).columns
    if len(non_numeric_cols) > 0:
        print(f"Warning: Non-numeric columns found and will be dropped: {list(non_numeric_cols)}")
        X = X.select_dtypes(include=np.number)

    print(f"\nTarget variable 'category' has {y.nunique()} unique classes.")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"Data split: {len(X_train)} training samples, {len(X_test)} testing samples.")

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    print("Features scaled using StandardScaler.")

    max_k = 20
    accuracies = []

    print(f"\nFinding best 'k' by testing values from 1 to {max_k}...")
    for k in range(1, max_k + 1):
        knn = KNeighborsClassifier(n_neighbors=k)

        knn.fit(X_train_scaled, y_train)

        y_pred = knn.predict(X_test_scaled)

        acc = accuracy_score(y_test, y_pred)
        accuracies.append(acc)

    plt.figure(figsize=(10, 6))
    plt.plot(range(1, max_k + 1), accuracies, marker='o', linestyle='dashed',
             color='blue', markersize=8)
    plt.title('KNN Accuracy vs. k-Value')
    plt.xlabel('k-Value (Number of Neighbors)')
    plt.ylabel('Test Accuracy')
    plt.grid(True)
    plt.xticks(range(1, max_k + 1))
    plt.tight_layout()
    plt.savefig('knn_accuracy_vs_k.png')
    print("Saved plot: 'knn_accuracy_vs_k.png'")

    best_accuracy = np.max(accuracies)
    best_k = np.argmax(accuracies) + 1

    print("\n--- Final Results ---")
    print(f"The best 'k' value (number of neighbors) is: {best_k}")
    print(f"The accuracy of the KNN algorithm at k={best_k} is: {best_accuracy:.4f}")

Successfully loaded 'teleCust.csv'

Dataset head:
   region  tenure  age  marital  address  income  ed  employ  retire  gender  \
0       2      13   44        1        9    64.0   4       5     0.0       0   
1       3      11   33        1        7   136.0   5       5     0.0       0   
2       3      68   52        1       24   116.0   1      29     0.0       1   
3       2      33   33        0       12    33.0   2       0     0.0       1   
4       2      23   30        1        9    30.0   1       2     0.0       0   

   reside  custcat  
0       2        1  
1       6        4  
2       2        3  
3       1        1  
4       4        3  

Error: Target column 'category' not found. Aborting.
